# 6. Psychometric Validation (Chapter 3.7)

This notebook justifies the aggregation of individual survey items into four composite dimension scores.

**Two analyses, two charts:**
1. **Cronbach's Alpha** – are items within each scale consistent enough to average?
2. **Inter-scale correlations** – are the four scales distinct enough to keep separate?

Outputs saved to `outputs/html/psychometrics/`.

## 1. Setup & Imports

In [ ]:
import logging
import sys
from pathlib import Path

import plotly.graph_objects as go
from IPython.display import display

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

from src.psychometrics_analysis import (
    load_long_format_data,
    build_item_matrices,
    build_reliability_summary,
    compute_scale_means,
    compute_inter_scale_correlations,
    RESILIENCE_SCALES,
)
from src.plot_config import OUTPUT_HTML_DIR

OUTPUT_DIR = project_root / OUTPUT_HTML_DIR / "psychometrics"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SCALE_COLORS = {
    "Bounce Back":    "#2E86AB",
    "Bounce Forward": "#A23B72",
    "Environment":    "#F18F01",
    "Mindset":        "#C73E1D",
}

## 2. Data Loading

In [ ]:
long_df  = load_long_format_data()
matrices = build_item_matrices(long_df)

## 3. Cronbach's Alpha

**Question:** Are the items within each scale consistent enough to average into a single score?

A Cronbach's Alpha of ≥ .70 is the accepted minimum for social science research.  
All four scales exceed this threshold, confirming that the composite scores are reliable.

In [ ]:
# Reliability table
reliability_df = build_reliability_summary(matrices)
display(
    reliability_df.style
    .format({"α": "{:.3f}"})
    .background_gradient(subset=["α"], cmap="RdYlGn", vmin=0.5, vmax=0.95)
    .set_caption("Table 3.7a – Internal Consistency (Cronbach's Alpha)")
)

In [ ]:
# Visualisation
plot_title  = "Cronbach's Alpha by Resilience Subscale"
x_axis_label = "Subscale"
y_axis_label = "Cronbach's α"
figure_height = 420

rel = reliability_df.loc[RESILIENCE_SCALES]

fig = go.Figure(go.Bar(
    x=rel.index.tolist(),
    y=rel["α"].tolist(),
    marker_color=[SCALE_COLORS[s] for s in rel.index],
    text=[f"α = {v:.3f}" for v in rel["α"]],
    textposition="outside",
    textfont_size=13,
))
fig.add_hline(y=0.70, line_dash="dash", line_color="#E67E22",
              annotation_text="Minimum threshold (0.70)", annotation_font_size=11)
fig.add_hline(y=0.80, line_dash="dot",  line_color="#27AE60",
              annotation_text="Good (0.80)", annotation_font_size=11)
fig.update_layout(
    title=plot_title,
    xaxis_title=x_axis_label,
    yaxis_title=y_axis_label,
    yaxis_range=[0, 1.05],
    height=figure_height,
    template="plotly_white",
    margin=dict(t=70, b=70),
)
fig.write_html(OUTPUT_DIR / "reliability_bars.html")
fig.show()

## 4. Inter-Scale Correlations

**Question:** Do the four scales measure different enough things to justify keeping them separate?

Moderate correlations (r ≈ .40–.75) are ideal — they confirm the scales are related (all tap into resilience) but not redundant.  
None exceed r = .85, so no two scales can be collapsed into one.

In [ ]:
# Inter-scale correlation heatmap
plot_title   = "Inter-Scale Pearson Correlations (*** p < .001)"
figure_height = 440
figure_width  = 520

scale_means_df = compute_scale_means(matrices)
r_df, p_df     = compute_inter_scale_correlations(scale_means_df)

# Build cell annotations: r value + significance stars
annot_text = []
for row_lbl in r_df.index:
    row_ann = []
    for col_lbl in r_df.columns:
        r_val = r_df.loc[row_lbl, col_lbl]
        p_val = p_df.loc[row_lbl, col_lbl]
        if row_lbl == col_lbl:
            row_ann.append("—")
        else:
            stars = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*"
            row_ann.append(f"r = {r_val:.3f}{stars}")
    annot_text.append(row_ann)

fig = go.Figure(go.Heatmap(
    z=r_df.values,
    x=r_df.columns.tolist(),
    y=r_df.index.tolist(),
    text=annot_text,
    texttemplate="%{text}",
    textfont_size=12,
    colorscale="RdBu_r",
    zmin=0.3, zmax=0.9,
    colorbar=dict(title="r"),
))
fig.update_layout(
    title=plot_title,
    height=figure_height,
    width=figure_width,
    template="plotly_white",
    margin=dict(t=70, b=40, l=140, r=40),
)
fig.write_html(OUTPUT_DIR / "inter_scale_correlations.html")
fig.show()

print("\nCorrelation matrix (r):")
display(r_df)

## 5. Summary (Chapter 3.7 text)

> To justify aggregating individual item responses into composite dimension scores, internal consistency was tested using Cronbach's Alpha. All four subscales exceeded the accepted threshold of α ≥ .70 — Bounce Back α = .838, Bounce Forward α = .770, Environment α = .843, Mindset α = .868 — confirming that items within each dimension measure a coherent construct and can be averaged into a single score.
>
> To verify that the four dimensions are genuinely distinct, inter-scale Pearson correlations were calculated. All correlations fell in the moderate range (r = .424 to r = .754, all p < .001), confirming that the four dimensions are related but non-redundant — justifying the use of four separate scores rather than a single resilience index.